### Pokemon - Data Wrangling
* Physical Cards, English Only, Secondary Market Price - Retail
* There will be nulls from the source data randomly in extCardType, marketPrice, and subTypeName.
    * Less than 1.7% of total cards at the most.

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

#### Define clean datetimes function.

In [14]:
# The purpose of this function is to convert errant release dates in source group data to a format pd.datetime can recognize, so it can convert to a datetime object.
# pd.datetime cannot convert after 7 decimals.
# For reference:
# https://docs.python.org/3/library/re.html
# re.sub(pattern, repl, string, count=0, flags=0)
def cleanAndParseDates(dateStr):
    if pd.isna(dateStr):
        return pd.NaT
    dateStr = re.sub(r'(\.\d{6})\d*Z$', r'\1', dateStr)  # trim after 6 digits, r'\1' references the first capture group in the first ()
    dateStr = re.sub(r'Z$', '', dateStr)  # remove Z if still present
    try:
        return pd.to_datetime(dateStr, utc=True) # parsing datetimes with mixed time zones will raise an error unless utc=True
    except Exception:
        return pd.NaT

#### Clean card data.

In [15]:
dfp = pd.read_excel('../data/dataPokemon/cardsPokemon.xlsx') # 10/15/25

# Standardize column name.
dfp = dfp.rename(columns = {"Source.Name" : "sourceName"})

# Specify the needed columns.
# extNumber will be used to identify single cards vs all other products (basic energy cards, boosters, code cards, figurines, decks, tins, etc.).
dfp = dfp[["sourceName", "productId", "cleanName", "groupId", "extNumber", "extRarity", "extCardType", "marketPrice", "subTypeName"]]

# First convert null to NaN then drop.
dfp['extNumber'].replace('', np.nan, inplace=True)
dfp.dropna(subset=['extNumber'], inplace = True)

# If needed as its own csv file, uncomment:
# dfp.to_csv("../data/dataPokemon/cardsPokemonClean.csv", index = False)

C:\Users\arsta\AppData\Local\Temp\ipykernel_22360\3217818265.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfp['extNumber'].replace('', np.nan, inplace=True)


#### Clean set group information.

In [16]:
dfpGroups = pd.read_csv('../data/dataPokemon/groupsPokemon.csv') # 10/15/25

dfpGroups = dfpGroups[["groupId", "name", "abbreviation", "publishedOn"]]

dfpGroups = dfpGroups.rename(columns = {"name" : "groupName", "abbreviation" : "groupCode", "publishedOn" : "releaseDate"})

# There are consistently 4 NaN Group Codes from origin data. 
# Since codes are universal and static once established, it is relatively safe to fill them in using precedent.
dfpGroups.loc[20, 'groupCode'] = 'MYFB'
dfpGroups.loc[31, 'groupCode'] = 'PPSC'
dfpGroups.loc[123, 'groupCode'] = 'MCD12'
dfpGroups.loc[129, 'groupCode'] = 'MCD11'

# Applying the function defined above.
dfpGroups['releaseDate'] = dfpGroups['releaseDate'].apply(cleanAndParseDates)

# Drop timezone (make native datetime) and normalize to remove time info
dfpGroups['releaseDate'] = dfpGroups['releaseDate'].dt.tz_convert(None).dt.normalize()

# create and insert a releaseYear column for plotting later
dfpGroups['releaseYear'] = dfpGroups['releaseDate'].dt.year

# If needed as its own csv file, uncomment:
# dfpGroups.to_csv("../data/dataPokemon/groupsPokemonClean.csv", index = False)

#### Merge cards with set groups.

In [17]:
dfp2 = pd.merge(dfp, dfpGroups, on = "groupId", how = "inner")

# Drop sourceName now that we have groupName.
dfp2.drop(columns = ["sourceName"], inplace = True)

# A more viewer-friendly order:
newOrderP = ['cleanName', 'groupId', 'groupName', 'productId', 'subTypeName', 'extCardType', 'extNumber', 'extRarity', 'releaseDate', 'releaseYear', 'marketPrice']
dfp2 = dfp2[newOrderP]

# Order by releaseDate, then groupName, then cleanName.
dfp2 = dfp2.sort_values(by=["releaseDate", "groupName", "cleanName"])

# Reset index after manipulation and to check new number of rows.
# Dropping the original index column.
dfp2 = dfp2.reset_index(drop = True)

# Make a pickle file for easier/safer imports.
dfp2.to_pickle("dfp2.pkl")

# If needed as its own csv file, uncomment:
# dfp2.to_csv("../data/dataPokemon/completePokemonClean.csv", index = False)

### Search check to ensure functionality.

In [18]:
dfp2[dfp2["cleanName"] == "Charizard"]

,cleanName,groupId,groupName,productId,subTypeName,extCardType,extNumber,extRarity,releaseDate,releaseYear,marketPrice
140,Charizard,604,Base Set,42382,Holofoil,Fire,004/102,Holo Rare,1999-01-09,1999,457.99
251,Charizard,1663,Base Set (Shadowless),106999,Unlimited Holofoil,Fire,004/102,Holo Rare,1999-01-09,1999,1923.19
836,Charizard,605,Base Set 2,42479,Holofoil,Fire,004/130,Holo Rare,2000-02-24,2000,304.88
2279,Charizard,1374,Legendary Collection,84196,Holofoil,Fire,003/110,Holo Rare,2002-05-24,2002,389.99
2280,Charizard,1374,Legendary Collection,84196,Reverse Holofoil,Fire,003/110,Holo Rare,2002-05-24,2002,NaN
3168,Charizard,1372,Skyridge,84186,Reverse Holofoil,Colorless,146/144,Secret Rare,2003-05-12,2003,NaN
3867,Charizard,1376,Dragon,84187,Holofoil,Fire,100/97,Secret Rare,2003-11-24,2003,739.97
6583,Charizard,1383,Power Keepers,84188,Holofoil,Fire,6/108,Holo Rare,2007-02-01,2007,137.52
6584,Charizard,1383,Power Keepers,84188,Reverse Holofoil,Fire,6/108,Holo Rare,2007-02-01,2007,119.09
7365,Charizard,1380,Secret Wonders,84189,Holofoil,Fire,3/132,Holo Rare,2007-11-07,2007,126.80
